<a href="https://colab.research.google.com/github/JoieLiu/PL-repo/blob/main/HW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil

In [ ]:
# 更新 SDK
!pip install -U -q google-generativeai

import google.generativeai as genai
from google.colab import userdata
import os

# 1. 注入金鑰 (請確保左側鑰匙圖示有名稱為 gemini_key 的項目)
try:
    api_key = userdata.get('gemini_key')
    genai.configure(api_key=api_key)
    os.environ["GOOGLE_API_KEY"] = api_key

    # 2. 【核心偵測】列出所有你可用的模型
    print("--- 系統偵測可用模型清單 ---")
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"✅ 可用: {m.name}")

except Exception as e:
    print(f"❌ 設定失敗：{e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


--- 系統偵測可用模型清單 ---
✅ 可用: models/gemini-2.5-flash
✅ 可用: models/gemini-2.5-pro
✅ 可用: models/gemini-2.0-flash
✅ 可用: models/gemini-2.0-flash-001
✅ 可用: models/gemini-2.0-flash-lite-001
✅ 可用: models/gemini-2.0-flash-lite
✅ 可用: models/gemini-2.5-flash-preview-tts
✅ 可用: models/gemini-2.5-pro-preview-tts
✅ 可用: models/gemma-3-1b-it
✅ 可用: models/gemma-3-4b-it
✅ 可用: models/gemma-3-12b-it
✅ 可用: models/gemma-3-27b-it
✅ 可用: models/gemma-3n-e4b-it
✅ 可用: models/gemma-3n-e2b-it
✅ 可用: models/gemma-4-26b-a4b-it
✅ 可用: models/gemma-4-31b-it
✅ 可用: models/gemini-flash-latest
✅ 可用: models/gemini-flash-lite-latest
✅ 可用: models/gemini-pro-latest
✅ 可用: models/gemini-2.5-flash-lite
✅ 可用: models/gemini-2.5-flash-image
✅ 可用: models/gemini-3-pro-preview
✅ 可用: models/gemini-3-flash-preview
✅ 可用: models/gemini-3.1-pro-preview
✅ 可用: models/gemini-3.1-pro-preview-customtools
✅ 可用: models/gemini-3.1-flash-lite-preview
✅ 可用: models/gemini-3-pro-image-preview
✅ 可用: models/nano-banana-pro-preview
✅ 可用: models/gemini-3.1-flas

In [ ]:
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
from google.colab import userdata
import os
import google.generativeai as genai

api_key = userdata.get('gemini_key')
os.environ["GOOGLE_API_KEY"] = api_key
os.environ["GEMINI_API_KEY"] = api_key # 兩個名字都設定，最保險
genai.configure(api_key=api_key)

print("✅ AI 鑰匙已就緒！")

✅ AI 鑰匙已就緒！


In [ ]:

SHEET_URL = "https://docs.google.com/spreadsheets/d/1AtRk7LqHWoJ7IoE7__2aU86lXr97NVNgJaCSDbk-phE/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
TIMEZONE = "Asia/Taipei"


In [ ]:
# --- 請確保執行這一格，定義所有工具函式 ---

def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update([header])
    return ws

def write_df(ws, df, header):
    if df.empty:
        ws.clear()
        ws.update([header])
        return
    # 補空值並轉字串，防止寫入失敗
    df_out = df.copy().fillna("").astype(str)
    ws.clear()
    ws.update([header] + df_out.values.tolist())

print("✅ 所有工具函式（ensure_worksheet, write_df）已定義完成！")

✅ 所有工具函式（ensure_worksheet, write_df）已定義完成！


In [ ]:
import pandas as pd


gsheets = gc.open_by_url(SHEET_URL)


try:
    sh = gc.open_by_url(SHEET_URL)
    print(f"✅ 成功連線至檔案：{sh.title}")
except Exception as e:
    print(f"❌ 無法透過網址開啟，請檢查 SHEET_URL 或權限：{e}")


TASKS_HEADER = [
    "id","task","status","priority","est_min","start_time","end_time",
    "actual_min","pomodoros","due_date","labels","notes",
    "created_at","updated_at","completed_at","planned_for"
]
LOGS_HEADER = [
    "log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"
]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]


ws_tasks = ensure_worksheet(sh, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(sh, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(sh, "web_clips", CLIPS_HEADER)

print("✅ 分頁檢查完成：tasks, pomodoro_logs, web_clips 已就緒")


✅ 成功連線至檔案：HW3
✅ 分頁檢查完成：tasks, pomodoro_logs, web_clips 已就緒


In [ ]:
# --- 修正後的連線方式 ---

# 直接使用你之前定義好的網址開啟檔案
try:
    sh = gc.open_by_url(SHEET_URL)
    print(f"✅ 成功連線至試算表檔案：{sh.title}")
except Exception as e:
    print(f"❌ 無法透過網址連線，請確認 SHEET_URL 是否正確：{e}")

# 執行這格後，原本的 NameError 和 APIError 500 都會消失

✅ 成功連線至試算表檔案：HW3


In [ ]:
def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    # 若沒有表頭就補上
    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update([header])
    return ws

TASKS_HEADER = [
    "id","task","status","priority","est_min","start_time","end_time",
    "actual_min","pomodoros","due_date","labels","notes",
    "created_at","updated_at","completed_at","planned_for"
]
LOGS_HEADER = [
    "log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"
]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

ws_tasks = ensure_worksheet(sh, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(sh, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(sh, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    # 保證欄位齊全
    for c in header:
        if c not in df.columns:
            df[c] = ""
    # 型別微調
    if "est_min" in df.columns:
        df["est_min"] = pd.to_numeric(df["est_min"], errors="coerce").fillna(0).astype(int)
    if "actual_min" in df.columns:
        df["actual_min"] = pd.to_numeric(df["actual_min"], errors="coerce").fillna(0).astype(int)
    if "pomodoros" in df.columns:
        df["pomodoros"] = pd.to_numeric(df["pomodoros"], errors="coerce").fillna(0).astype(int)
    return df[header]


def write_df(ws, df, header):
    if df is None: return
    # 補空值並轉字串
    df_out = df.copy().fillna("").astype(str)
    data = [header] + df_out.values.tolist()

    ws.clear() # 清除舊的
    ws.update(data) # <--- 這行最重要！沒這行就沒存入雲端
    print("✅ 資料已推送到 Google Sheets")


def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs, LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )

tasks_df, logs_df, clips_df = refresh_all()

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8],
        "task": task.strip(),
        "status": "todo",
        "priority": priority or "M",
        "est_min": int(est_min) if est_min else 25,
        "start_time": "",
        "end_time": "",
        "actual_min": 0,
        "pomodoros": 0,
        "due_date": due_date or "",
        "labels": labels or "",
        "notes": notes or "",
        "created_at": _now,
        "updated_at": _now,
        "completed_at": "",
        "planned_for": planned_for or ""  # 可填 today / tomorrow / 空白
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 已新增任務", tasks_df


def update_task_status(task_id, new_status):
    global tasks_df
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx)==0:
        return "⚠️ 找不到任務", tasks_df
    i = idx[0]
    tasks_df.loc[i, "status"] = new_status
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()
    if new_status == "done" and not tasks_df.loc[i, "completed_at"]:
        tasks_df.loc[i, "completed_at"] = tznow().isoformat()
    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    return "✅ 狀態已更新", tasks_df

def mark_done(task_id):
    return update_task_status(task_id, "done")

def recalc_task_actuals(task_id):
    """根據 logs_df 回寫 actual_min 與 pomodoros"""
    global tasks_df, logs_df
    work_logs = logs_df[(logs_df["task_id"]==task_id) & (logs_df["phase"]=="work")]
    total_min = work_logs["minutes"].astype(float).sum() if not work_logs.empty else 0
    pomos = int(round(total_min / 25.0))
    idx = tasks_df.index[tasks_df["id"]==task_id]
    if len(idx)==0: return
    i = idx[0]
    tasks_df.loc[i,"actual_min"] = int(total_min)
    tasks_df.loc[i,"pomodoros"] = pomos
    tasks_df.loc[i,"updated_at"] = tznow().isoformat()

def list_task_choices():
    global tasks_df
    if tasks_df.empty:
        return []
    # 顯示： [status] (P:priority) task  — id
    def row_label(r):
        return f"[{r['status']}] (P:{r['priority']}) {r['task']} — {r['id']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

# 我們採「按鈕開始 / 結束」模式（避免後端阻塞），每次按「開始」會先記住 start_ts，
# 按「結束」時計算分鐘並寫入 logs，再回填任務 actual_min / pomodoros。

_active_sessions = {}  # { task_id: {"phase": "work"/"break", "start_ts": iso, "cycles": int} }

def start_phase(task_id, phase, cycles):
    if not task_id: return "⚠️ 請先選擇任務"
    _active_sessions[task_id] = {
        "phase": phase,
        "start_ts": tznow().isoformat(),
        "cycles": int(cycles) if cycles else 1
    }
    return f"▶️ 已開始：{phase}（task: {task_id}）"

def end_phase(task_id, note):
    global logs_df, tasks_df
    if task_id not in _active_sessions:
        return "⚠️ 尚未開始任何階段"
    sess = _active_sessions.pop(task_id)
    start = pd.to_datetime(sess["start_ts"])
    end = tznow()
    minutes = round((end - start).total_seconds() / 60.0, 2)
    log = pd.DataFrame([{
        "log_id": str(uuid.uuid4())[:8],
        "task_id": task_id,
        "phase": sess["phase"],
        "start_ts": start.isoformat(),
        "end_ts": end.isoformat(),
        "minutes": minutes,
        "cycles": int(sess["cycles"]),
        "note": note or ""
    }])
    logs_df = pd.concat([logs_df, log], ignore_index=True)
    write_df(ws_logs, logs_df, LOGS_HEADER)

    # 回填任務
    if sess["phase"] == "work":
        recalc_task_actuals(task_id)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)

    return f"⏹️ 已結束：{sess['phase']}，紀錄 {minutes} 分鐘"

def generate_today_plan():
    global tasks_df
    print("--- 執行 generate_today_plan ---")
    if tasks_df is None or tasks_df.empty:
        print("❌ 任務資料為空")
        return "❌ 目前沒有任務資料，請先新增任務或按下『重新整理』。"

    # 1. 取得日期與寬鬆篩選
    now = tznow()
    today_str = now.date().isoformat()

    # 確保資料格式正確
    tasks_df["due_date"] = tasks_df["due_date"].astype(str).fillna("")
    tasks_df["planned_for"] = tasks_df["planned_for"].astype(str).fillna("")

    mask = (
        (tasks_df["due_date"].str.contains(today_str, na=False)) |
        (tasks_df["planned_for"].str.lower().str.contains("today", na=False))
    ) & (tasks_df["status"] != "done")

    cand = tasks_df[mask].copy()
    print(f"今日候選任務數量: {len(cand)}")

    if cand.empty:
        return f"📭 今天（{today_str}）找不到待辦任務。\n請確認 Due Date 是否設為 {today_str}。"

    # 2. 排序 (H > M > L)
    pr_order = {"H": 0, "M": 1, "L": 2}
    cand["p_ord"] = cand["priority"].map(pr_order).fillna(3)
    cand = cand.sort_values(["p_ord", "est_min"], ascending=[True, True])

    # 3. 嘗試呼叫 AI (第一層：最強大腦)
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    plan_md = ""
    print(f"API Key 已設定: {bool(api_key)}")

    if api_key:
        try:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            # 將模型改為更穩定的 gemini-2.5-flash
            model = genai.GenerativeModel("gemini-2.5-flash")
            print(f"嘗試使用模型: {model.model_name}")

            items_json = cand[["id", "task", "est_min", "priority"]].to_json(orient="records", force_ascii=False)
            prompt = f"你是一位排程助手，請根據以下任務 JSON 安排早中晚計畫：{items_json}"
            print(f"AI 提示詞: {prompt[:200]}...")

            print("--- 呼叫 AI 生成內容中 ---") # Added debug print
            resp = model.generate_content(prompt)
            print("--- AI 生成內容完成 ---")  # Added debug print

            print(f"AI 回應內容 (部分): {resp.text[:200]}...")
            plan_md = "### 🧠 AI 智慧排程結果\n" + resp.text

        except Exception as e:
            print(f"⚠️ AI 連線失敗或產生內容錯誤：{e}")
            # 第二層：AI 故障時的「保險分析」
            total_min = cand["est_min"].sum()
            h_priority_count = len(cand[cand["priority"] == "H"])

            if total_min > 480:
                advice = "❌ 警告：今日預估工時超過 8 小時，請務必精簡任務。"
            elif h_priority_count > 3:
                advice = "⚠️ 警告：高優先級任務過多，建議專注處理前三項。"
            else:
                advice = "基礎配置紮實，請依照節奏執行，記得定時休息。"

            plan_md = f"""### 📋 AI 離線 - 自動排程分析\n> 偵測到 AI 連線異常（{e}），已自動切換至本機數據分析。\n\n- **今日負載：** {total_min} 分鐘 / {len(cand)} 項任務\n- **壓力評估：** {"偏高" if total_min > 300 else "適中"}\n- **核心建議：** {advice}\n"""
    else:
        # 第三層：連鑰匙都沒給時的提示
        plan_md = "❌ 系統未偵測到 API Key，請先執行金鑰注入區塊（Cell）。"

    # 4. 規則式排程（作為底部對照）
    buckets = {"Morning": [], "Afternoon": [], "Evening": []}
    for i, (_, r) in enumerate(cand.iterrows()):
        target = list(buckets.keys())[i % 3]
        buckets[target].append(f"- [{r['id']}] {r['task']} ({r['est_min']} min)")

    rule_md = "### 📋 規則式清單參考\n"
    for k, v in buckets.items():
        rule_md += f"#### {k}\n" + ("\n".join(v) if v else "（無任務）") + "\n"

    final_output = (plan_md + "\n\n---\n\n" + rule_md).strip()
    print("--- generate_today_plan 執行結束 ---")
    return final_output

# 今日完成率
def today_summary():
    global tasks_df
    now = tznow()
    today_str = now.date().isoformat()

    # Ensure columns are string types for consistent filtering
    tasks_df["due_date"] = tasks_df["due_date"].astype(str).fillna("")
    tasks_df["planned_for"] = tasks_df["planned_for"].astype(str).fillna("")

    planned_tasks = tasks_df[
        ((tasks_df["due_date"].str.contains(today_str, na=False)) |
         (tasks_df["planned_for"].str.lower().str.contains("today", na=False)))
    ]
    done_tasks = planned_tasks[planned_tasks["status"]=="done"]
    total = len(planned_tasks)
    done_n = len(done_tasks)
    rate = (done_n/total*100) if total>0 else 0
    return f"📅 今日計畫任務：{total}；✅ 完成：{done_n}；📈 完成率：{rate:.1f}%"

# =========================
# 爬蟲：擷取文字或連結並可加入任務
# =========================


def crawl_and_convert(url):
    try:
        # --- 第一階段：網頁抓取 ---
        resp = requests.get(url, timeout=15, headers={"User-Agent":"Mozilla/5.0"})
        resp.encoding = 'utf-8' # 確保中文不亂碼
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # 抓取網頁標題與主要文字 (去除腳本與樣式)
        for script in soup(["script", "style"]):
            script.extract()
        title = soup.title.string if soup.title else "未知公告"
        body_text = soup.get_text(separator=' ', strip=True)[:2000] # 抓取前 2000 字供 AI 分析

        # --- 第二階段：Gemini AI 語義分析 (全自動路徑匹配) ---
        api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
        if not api_key: return "❌ 未偵測到 API Key"

        genai.configure(api_key=api_key)

        # 準備測試的模型清單，按優先順序排列
        model_names = ["gemini-3-flash-preview", "gemini-2.0-flash", "gemini-flash-latest","gemini-pro-latest"]
        resp = None
        final_model_used = ""

        prompt = f"""
        你是一位課程助理。請分析以下網頁公告，判斷是否有「作業、考試、繳交期限」。
        標題：{title}
        內容：{body_text}

        若有，請嚴格以 JSON 回傳（不可有其他文字）：
        {{
            "is_task": true,
            "task_name": "任務簡稱",
            "due_date": "YYYY-MM-DD (無則留空)",
            "priority": "H/M/L",
            "notes": "要求摘要"
        }}
        若無，回傳:{{"is_task": false}}
        """

        # 自動循環嘗試，直到有一個模型能動
        for m_name in model_names:
            try:
                model = genai.GenerativeModel(m_name)
                resp = model.generate_content(prompt)
                if resp:
                    final_model_used = m_name
                    break
            except Exception:
                continue # 如果這個報 404，就試下一個

        if not resp:
            return "❌ AI 嘗試了所有路徑 (Flash/Pro) 均回報 404，請檢查 API Key 權限或稍後再試。"

        # --- 第三階段：解析與寫入 (這部分維持原樣) ---
        json_match = re.search(r'\{.*\}', resp.text, re.DOTALL)
        if not json_match: return "⚠️ AI 回傳內容非標準格式。"

        res = json.loads(json_match.group())

        # --- 第三階段：自動寫入 Google Sheet ---
        if res.get("is_task"):
            new_id = str(uuid.uuid4())[:8]
            now_iso = tznow().isoformat()

            # 依照你的 Google Sheet 格式組合資料
            new_row = [
                new_id,
                res['task_name'],
                "todo",
                res['priority'],
                25, # 預設工時
                "", "", 0, 0,
                res['due_date'],
                "AI自動擷取",
                f"來源網址：{url}\n備註：{res['notes']}",
                now_iso, now_iso, "", "today"
            ]

            new_task_df = pd.DataFrame([{
                "id": new_id,
                "task": res['task_name'],
                "status": "todo",
                "priority": res['priority'],
                "est_min": 25,
                "start_time": "",
                "end_time": "",
                "actual_min": 0,
                "pomodoros": 0,
                "due_date": res['due_date'],
                "labels": "AI自動擷取",
                "notes": f"來源網址：{url}\n備註：{res['notes']}",
                "created_at": now_iso,
                "updated_at": now_iso,
                "completed_at": "",
                "planned_for": "today"
            }])
            global tasks_df
            tasks_df = pd.concat([tasks_df, new_task_df], ignore_index=True)
            write_df(ws_tasks, tasks_df, TASKS_HEADER)

            return f"✅ 已成功從網頁自動新增任務：\n📌 {res['task_name']} (截止：{res['due_date'] if res['due_date'] else '未註明'})"
        else:
            return "ℹ️ 分析完成：此公告內容中似乎不包含待辦事項。"

    except Exception as e:
        return f"⚠️ 轉換失敗：{e}"

def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df
    if not clip_ids:
        return "⚠️ 請先勾選要加入的爬蟲項目", clips_df, tasks_df
    sel = clips_df[clips_df["clip_id"].isin(clip_ids)]
    _now = tznow().isoformat()
    new_tasks = []
    for _, r in sel.iterrows():
        title = r["text"] or r["href"] or "（未命名）"
        note = f"來源：{r['url']}\n選擇器：{r['selector']}\n連結：{r['href']}"
        new_tasks.append({
            "id": str(uuid.uuid4())[:8],
            "task": title[:120],
            "status": "todo",
            "priority": default_priority or "M",
            "est_min": int(est_min) if est_min else 25,
            "start_time": "",
            "end_time": "",
            "actual_min": 0,
            "pomodoros": 0,
            "due_date": "",
            "labels": "from:crawler",
            "notes": note,
            "created_at": _now,
            "updated_at": _now,
            "completed_at": "",
            "planned_for": ""
        })
    if new_tasks:
        tasks_df = pd.concat([tasks_df, pd.DataFrame(new_tasks)], ignore_index=True)
        # 標記已加入
        clips_df.loc[clips_df["clip_id"].isin(clip_ids), "added_to_task"] = "yes"
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        write_df(ws_clips, clips_df, CLIPS_HEADER)
        return f"✅ 已加入 {len(new_tasks)} 項為任務", clips_df, tasks_df
    return "⚠️ 無可加入項目", clips_df, tasks_df

# =========================
# Gradio 介面
# =========================
def _refresh():
    global tasks_df, logs_df, clips_df
    # 1. 從雲端讀取最新資料
    tasks_df, logs_df, clips_df = refresh_all()

    # 2. 重新產生選單的選項
    new_choices = list_task_choices()

    # 3. 取得今日分析摘要
    summ = today_summary()

    # 4. 回傳給所有對應的元件 (請注意這裡回傳了 7 個值，剛好對應你的 outputs)
    return (
        tasks_df,      # 對應 grid_tasks
        logs_df,       # 對應 grid_logs
        clips_df,      # 對應 grid_clips
        new_choices,   # 對應 task_choice (分頁1的選單)
        new_choices,   # 對應 sel_task (分頁2的選單)
        summ,          # 對應 out_summary (摘要1)
        summ           # 對應 out_summary2 (摘要2)
    )

# The functions 'crawl' and 'url' are not defined in the provided context.
# I'll define dummy functions and a placeholder for 'url' to make the code runnable
# This assumes 'url' was intended to be 'url_ai' based on the 'Crawler' tab content.
# And 'crawl' function should take (url, selector, mode, limit) and return (df, msg).
# The previous `btn_crawl.click` was using `url_ai`, so I'll adjust it.
def crawl(url, selector, mode, limit):
    # Dummy implementation for crawl function
    print(f"Crawling {url} with selector {selector}, mode {mode}, limit {limit}")
    # Return an empty dataframe and a message as a placeholder
    return pd.DataFrame(columns=CLIPS_HEADER), "Dummy crawl message: No actual crawling performed."

# The function 'add_clips_to_tasks' is not defined in the provided context.
# I'll assume it was a typo and intended to be 'add_clips_as_tasks'.
# The arguments also differ slightly: add_clips_as_tasks returns (msg, new_clips, new_tasks).
# The original binding in the notebook was: btn_add_clips.click(add_clips_to_tasks, [clip_ids, default_priority, clip_est], [msg_add_clips])
# I will adapt this to the existing `add_clips_as_tasks` function.
def add_clips_to_tasks(clip_ids_str, pr, est):
    # This function acts as a wrapper for add_clips_as_tasks to match the expected outputs from the Gradio binding
    ids = [c.strip() for c in (clip_ids_str or "").split(",") if c.strip()]
    msg, new_clips, new_tasks = add_clips_as_tasks(ids, pr, est)
    return msg # Returning only msg_add_clips, as per the original binding output.

def _crawl_and_save(u, s, m, l):
    df, msg = crawl(u, s, m, l)
    # 寫入 web_clips（覆蓋式追加：合併舊資料）
    global clips_df
    if not df.empty:
        clips_df = pd.concat([clips_df, df], ignore_index=True)
        write_df(ws_clips, clips_df, CLIPS_HEADER)
    return msg, clips_df

def _add_clips(clip_ids_str, pr, est):
    ids = [c.strip() for c in (clip_ids_str or "").split(",") if c.strip()]
    msg, new_clips, new_tasks = add_clips_as_tasks(ids, pr, est)
    return msg, new_clips, new_tasks

with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫（Sheet/Gradio/爬蟲）") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘（Google Sheet＋Gradio＋Crawler＋AI 計畫）")
    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())
        out_summary2 = gr.Markdown(today_summary())

    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="寫 HW3 報告 / 修正 SQL / …")
                priority = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（逗號分隔，可空白）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="任務清單（直接從 Sheet 來）", interactive=False)


        with gr.Row():
            # 加上 allow_custom_value=True 可以消除那個討厭的黃色警告
            task_choice = gr.Dropdown(
                choices=list_task_choices(),
                label="選取任務（用於更新）",
                allow_custom_value=True
            )
            new_status = gr.Dropdown(
                ["todo", "in-progress", "done"],
                value="in-progress",
                label="更新狀態"
            )

        # 建議把按鈕放在獨立的一列，看起來比較專業
        with gr.Row():
            btn_update = gr.Button("✏️ 更新狀態", variant="secondary")
            btn_done = gr.Button("✅ 直接標記完成", variant="primary") # Primary 會變成藍色比較醒目

        msg_update = gr.Markdown()


    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(
                choices=list_task_choices(),
                label="選擇任務",
                allow_custom_value=True
            )
            cycles = gr.Number(value=1, precision=0, label="番茄數（僅作紀錄）")

        with gr.Row():
            with gr.Column():
                btn_start_work = gr.Button("▶️ 開始工作") # 預設灰色
                note_work = gr.Textbox(label="工作備註", placeholder="這段時間要做什麼？")
                btn_end_work = gr.Button("⏹️ 結束工作並記錄")

            with gr.Column():
                btn_start_break = gr.Button("🍵 開始休息")
                note_break = gr.Textbox(label="休息備註", placeholder="休息時的心得？")
                btn_end_break = gr.Button("⏹️ 結束休息並記錄")

        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df, label="番茄鐘紀錄", interactive=False)

    with gr.Tab("AI Plan"):
        gr.Markdown("把**今天的任務**排成 **morning / afternoon / evening** 三段行動計畫。若未設 GEMINI_API_KEY，會用規則式。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    with gr.Tab("Crawler"):
        gr.Markdown("### 🎓 課程公告 AI 自動轉任務")
        with gr.Row():
            url_ai = gr.Textbox(
                label="公告網址",
                placeholder="貼上課程公告、作業說明網頁網址...",
                scale=4
            )
            btn_ai_convert = gr.Button("🧠 AI 智慧轉換", variant="primary", scale=1)

        msg_ai_status = gr.Markdown("系統將自動分析網頁內容，判斷是否有作業、截止日期並自動加入清單。")

        gr.HTML("<hr>") # 分隔線

        with gr.Accordion("🔧 進階手動擷取工具 (原本的功能)", open=False):
            selector = gr.Textbox(label="CSS Selector", placeholder="a.news-item / h2.title")
            mode = gr.Radio(["text","href","both"], value="text", label="擷取內容")
            limit = gr.Number(value=20, precision=0, label="最多擷取幾筆")
            btn_crawl = gr.Button("🕷️ 開始擷取")
            msg_crawl = gr.Markdown()
            grid_clips = gr.Dataframe(value=clips_df, label="擷取結果", interactive=True)

            with gr.Row():
                clip_ids = gr.Textbox(label="要加入任務的 clip_id")
                default_priority = gr.Dropdown(["H","M","L"], value="L", label="優先級")
                clip_est = gr.Number(value=25, precision=0, label="預估分鐘")

            btn_add_clips = gr.Button("➕ 將擷取項目加入為任務")
            msg_add_clips = gr.Markdown()

        # --- 綁定 AI 轉換動作 ---
        btn_ai_convert.click(
            crawl_and_convert, # 這是我們剛才寫的有 AI 邏輯的函式
            inputs=[url_ai],
            outputs=[msg_ai_status]
        ).then(
            _refresh, # 成功後自動刷新所有表格與選單
            outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
        )

    with gr.Tab("Summary"):
        gr.Markdown("### 📈 個人產能分析報表")
        btn_summary = gr.Button("📊 重新計算今日完成率", variant="primary")

        # 這裡一定要定義 out_summary2，它是專屬於這個分頁的顯示框
        out_summary2 = gr.Markdown()

        # 綁定動作：點擊按鈕時，呼叫 today_summary，並將結果輸出到上面的框
        btn_summary.click(
            fn=today_summary,
            inputs=None,
            outputs=out_summary2
        )

    # === 綁定動作 ===

    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2])

    btn_add.click(
        add_task,
        inputs=[task, priority, est_min, due_date, labels, notes, planned_for],
        outputs=[msg_add, grid_tasks]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_update.click(
        update_task_status,
        inputs=[task_choice, new_status],
        outputs=[msg_update, grid_tasks]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_done.click(
        mark_done,
        inputs=[task_choice],
        outputs=[msg_update, grid_tasks]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_start_work.click(
        start_phase, inputs=[sel_task, gr.State("work"), cycles], outputs=[msg_pomo]
    )
    btn_end_work.click(
        end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )
    btn_start_break.click(
        start_phase, inputs=[sel_task, gr.State("break"), cycles], outputs=[msg_pomo]
    )
    btn_end_break.click(
        end_phase, inputs=[sel_task, note_break], outputs=[msg_pomo]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_plan.click(
        fn=generate_today_plan,
        inputs=None,
        outputs=out_plan
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_crawl.click(_crawl_and_save, inputs=[url_ai, selector, mode, limit], outputs=[msg_crawl, grid_clips]) \
        .then(
            _refresh,
            outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
        )

    btn_add_clips.click(
        _add_clips,
        inputs=[clip_ids, default_priority, clip_est],
        outputs=[msg_add_clips, grid_clips, grid_tasks]
    ).then(
        _refresh,
        outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
    )

    btn_summary.click(
            fn=_refresh,
            outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary, out_summary2]
        )


# 順便開啟「除錯模式」，這樣如果 AI 出錯，我們可以直接在網頁上看到紅字原因
    demo.queue().launch(debug=True, share=True, inline=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4d7a1398a69d2514e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 